# Random Forest — From Scratch
> Ensemble of Decision Trees with Bootstrap + Feature Subsampling.

**Uses the DecisionTree built from scratch above (re-implemented here).**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.random.seed(42)

## Reuse Decision Tree (self-contained copy)

In [ ]:
def _mode(arr):
    vals, counts = np.unique(arr, return_counts=True)
    return vals[np.argmax(counts)]

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature=feature; self.threshold=threshold
        self.left=left; self.right=right; self.value=value

class DecisionTree:
    def __init__(self, max_depth=10, max_features=None):
        self.max_depth=max_depth; self.max_features=max_features

    def _gini(self, y):
        _, c = np.unique(y, return_counts=True); p = c/len(y); return 1-np.sum(p**2)

    def _best_split(self, X, y):
        n, d = X.shape
        feat_idx = (np.random.choice(d, self.max_features, replace=False)
                    if self.max_features else range(d))
        bg, bf, bt = float('inf'), None, None
        for f in feat_idx:
            for t in np.unique(X[:,f]):
                l, r = y[X[:,f]<=t], y[X[:,f]>t]
                if not len(l) or not len(r): continue
                g = (len(l)*self._gini(l)+len(r)*self._gini(r))/n
                if g < bg: bg,bf,bt = g,f,t
        return bf, bt

    def _build(self, X, y, depth):
        if depth>=self.max_depth or len(np.unique(y))==1 or len(y)<2:
            return Node(value=_mode(y))
        f,t = self._best_split(X,y)
        if f is None: return Node(value=_mode(y))
        m = X[:,f]<=t
        return Node(f,t, self._build(X[m],y[m],depth+1), self._build(X[~m],y[~m],depth+1))

    def fit(self, X, y): self.root=self._build(np.array(X),np.array(y),0)

    def _trav(self, n, x):
        if n.value is not None: return n.value
        return self._trav(n.left,x) if x[n.feature]<=n.threshold else self._trav(n.right,x)

    def predict(self, X): return np.array([self._trav(self.root,x) for x in X])

## Random Forest Class

In [ ]:
class RandomForest:
    def __init__(self, n_trees=50, max_depth=10, max_features='sqrt'):
        self.n_trees=n_trees; self.max_depth=max_depth; self.max_features=max_features

    def fit(self, X, y):
        X,y = np.array(X), np.array(y)
        n, d = X.shape
        mf = int(np.sqrt(d)) if self.max_features=='sqrt' else int(np.log2(d))
        self.trees = []
        for _ in range(self.n_trees):
            idx = np.random.choice(n, n, replace=True)  # bootstrap
            tree = DecisionTree(max_depth=self.max_depth, max_features=mf)
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)

    def predict(self, X):
        preds = np.array([t.predict(X) for t in self.trees])  # (n_trees, n_samples)
        return np.array([_mode(preds[:,i]) for i in range(preds.shape[1])])

## Train on Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForest(n_trees=50, max_depth=8)
rf.fit(X_tr, y_tr)
print(f"Accuracy: {accuracy_score(y_te, rf.predict(X_te))*100:.2f}%")

## Accuracy vs Number of Trees

In [ ]:
accs = []
ns = [1, 5, 10, 20, 30, 50, 75, 100]
for n in ns:
    m = RandomForest(n_trees=n, max_depth=8); m.fit(X_tr, y_tr)
    accs.append(accuracy_score(y_te, m.predict(X_te)))

plt.figure(figsize=(8,4))
plt.plot(ns, accs, 'o-', color='seagreen')
plt.xlabel("Number of Trees"); plt.ylabel("Accuracy")
plt.title("Random Forest – Trees vs Accuracy"); plt.grid(True)
plt.tight_layout(); plt.show()